# Publication tables

Generate the ten tables covering optimizer settings, embedding dimensions, proposal rankings, external validation, geometric agreement and method/architecture comparisons. The choice-head ablation table is generated in notebook 18.

**Inputs:** included analysis outputs and comparison CSVs, declared below. **Outputs:** CSV and LaTeX tables in `outputs/manuscript_tables/`. Run in a fresh kernel from the repository root or `notebooks/`.

Tables are identified by content and descriptive filenames, not by a manuscript-specific supplementary number. This notebook formats the supplied benchmark results without refitting their models or changing their numerical values. The optimizer experiments use the single-score exponential head, whereas the ideal-point benchmarks use the flexible four-branch head. Neither is the positive symmetric model used in the main maps.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "data/raw").is_dir(), "Open from the project root or notebooks directory."
PROJECT = ROOT
INPUTS = {
    "optimizers": PROJECT / "data/benchmarks/optimizer_comparison.csv",
    "learning_rates": PROJECT / "data/benchmarks/learning_rate_comparison.csv",
    "dimensions": PROJECT / "outputs/diagnostics/Chile_dimensionality.csv",
    "rankings": PROJECT / "data/processed/proposal_rankings.csv",
    "proposal_labels": PROJECT / "data/raw/proposal_english_labels.csv",
    "validation": PROJECT / "outputs/validation/validation_summary.csv",
    "geometry": PROJECT / "outputs/tables/geometry_summary.csv",
    "architectures": PROJECT / "data/benchmarks/architecture_results.csv",
}
missing = [str(path.relative_to(PROJECT)) for path in INPUTS.values() if not path.is_file()]
assert not missing, "The included analysis inputs are missing: " + ", ".join(missing)
OUT = PROJECT / "outputs/manuscript_tables"
OUT.mkdir(parents=True, exist_ok=True)

## Optimizers and learning rates

The optimizer comparison uses nine supplied results at learning rate 0.001. The learning-rate comparison uses Adam and Adamax at 0.1, 0.01 and 0.001. Their shared 0.001 rows must agree exactly. These all-cycle experiments evaluate the single-score exponential choice head.

In [2]:
optimizer_source = pd.read_csv(INPUTS["optimizers"])
learning_rate_source = pd.read_csv(INPUTS["learning_rates"])
required = ["Optimizer", "Learning rate", "Held-out accuracy"]
assert list(optimizer_source.columns) == required
assert list(learning_rate_source.columns) == required
optimizer_order = ["SGD", "RMSprop", "Adam", "AdamW", "Adadelta", "Adagrad", "Adamax", "Adafactor", "Nadam"]
assert len(optimizer_source) == len(optimizer_order)
assert optimizer_source["Optimizer"].is_unique
assert set(optimizer_source["Optimizer"]) == set(optimizer_order)
assert np.allclose(optimizer_source["Learning rate"], 0.001, rtol=0, atol=1e-15)
assert optimizer_source["Held-out accuracy"].between(0, 1).all()
optimizer_table = optimizer_source.set_index("Optimizer").loc[optimizer_order].reset_index()
display(optimizer_table)
optimizer_table.to_csv(OUT / "optimizer_comparison.csv", index=False)
optimizer_table.to_latex(OUT / "optimizer_comparison.tex", index=False, escape=True,
                         column_format="lrr", float_format="%.3f")

expected_rate_pairs = {(optimizer, rate) for optimizer in ["Adam", "Adamax"] for rate in [0.1, 0.01, 0.001]}
assert len(learning_rate_source) == len(expected_rate_pairs)
assert set(zip(learning_rate_source["Optimizer"], learning_rate_source["Learning rate"])) == expected_rate_pairs
assert not learning_rate_source.duplicated(["Optimizer", "Learning rate"]).any()
assert learning_rate_source["Held-out accuracy"].between(0, 1).all()
for optimizer in ["Adam", "Adamax"]:
    from_optimizers = optimizer_source.loc[optimizer_source["Optimizer"].eq(optimizer), "Held-out accuracy"].iloc[0]
    from_rates = learning_rate_source.loc[learning_rate_source["Optimizer"].eq(optimizer)
        & learning_rate_source["Learning rate"].eq(0.001), "Held-out accuracy"].iloc[0]
    assert np.isclose(from_optimizers, from_rates, rtol=0, atol=1e-12)
learning_rate_table = learning_rate_source.sort_values(["Learning rate", "Optimizer"], ascending=[False, True]).reset_index(drop=True)
display(learning_rate_table)
learning_rate_table.to_csv(OUT / "learning_rate_comparison.csv", index=False)
learning_rate_table.to_latex(OUT / "learning_rate_comparison.tex", index=False, escape=True,
                            column_format="lrr", float_format="%.3f")

,Optimizer,Learning rate,Held-out accuracy
0,SGD,0.001,0.503815
1,RMSprop,0.001,0.665559
2,Adam,0.001,0.694347
3,AdamW,0.001,0.685624
4,Adadelta,0.001,0.503815
5,Adagrad,0.001,0.503815
6,Adamax,0.001,0.686133
7,Adafactor,0.001,0.697623
8,Nadam,0.001,0.664488


,Optimizer,Learning rate,Held-out accuracy
0,Adam,0.100,0.562336
1,Adamax,0.100,0.660357
2,Adam,0.010,0.667145
3,Adamax,0.010,0.671120
4,Adam,0.001,0.694347
5,Adamax,0.001,0.686133


## Embedding dimensions

Use the fixed-seed first-cycle comparison in one, two, three, four and ten dimensions. Loss and accuracy use all held-out records. The LaTeX table uses three decimals, while its CSV retains the source precision.

In [3]:
dimension_source = pd.read_csv(INPUTS["dimensions"])
dimension_columns = ["dimension", "test_loss_all_records", "test_accuracy_all_records"]
assert list(dimension_source.columns) == dimension_columns
assert len(dimension_source) == 5 and dimension_source["dimension"].is_unique
assert set(dimension_source["dimension"]) == {1, 2, 3, 4, 10}
assert np.isfinite(dimension_source[dimension_columns]).all().all()
assert dimension_source["test_loss_all_records"].ge(0).all()
assert dimension_source["test_accuracy_all_records"].between(0, 1).all()
dimension_table = dimension_source.sort_values("dimension").reset_index(drop=True).rename(columns={
    "dimension": "Embedding dimensions", "test_loss_all_records": "Held-out loss",
    "test_accuracy_all_records": "Held-out accuracy"})
display(dimension_table)
dimension_table.to_csv(OUT / "embedding_dimensions.csv", index=False)
dimension_table.to_latex(OUT / "embedding_dimensions.tex", index=False, escape=True,
                         column_format="rrr", float_format="%.3f")

,Embedding dimensions,Held-out loss,Held-out accuracy
0,1,0.608686,0.650666
1,2,0.594979,0.680987
2,3,0.566223,0.703317
3,4,0.560734,0.708376
4,10,0.556558,0.717668


## Proposals associated with each ideological group

For the left-associated list, sort the left/right rank ratio in ascending order. Reverse the ratio for the right-associated list. Proposal identifiers break ties. English labels are checked against the source wording before export.

In [4]:
rankings = pd.read_csv(INPUTS["rankings"])
labels = pd.read_csv(INPUTS["proposal_labels"])
rank_columns = ["id", "short_name", "name", "Ranking Izquierda", "Ranking Derecha",
                "ranking_izquierda_sort", "ranking_derecha_sort"]
assert set(rank_columns).issubset(rankings.columns)
assert {"id", "spanish_label", "spanish_proposal_wording", "english_label"}.issubset(labels.columns)
assert rankings["id"].is_unique and labels["id"].is_unique
assert len(rankings) >= 10
assert rankings[["Ranking Izquierda", "Ranking Derecha"]].gt(0).all().all()
assert np.allclose(rankings["ranking_izquierda_sort"], rankings["Ranking Izquierda"] / rankings["Ranking Derecha"], rtol=1e-12, atol=1e-12)
assert np.allclose(rankings["ranking_derecha_sort"], rankings["Ranking Derecha"] / rankings["Ranking Izquierda"], rtol=1e-12, atol=1e-12)
left = rankings.sort_values(["ranking_izquierda_sort", "id"], kind="mergesort").head(10)
right = rankings.sort_values(["ranking_derecha_sort", "id"], kind="mergesort").head(10)
selected_ids = set(left["id"]) | set(right["id"])
assert selected_ids.issubset(set(labels["id"])), "An English label is missing for a newly selected proposal."
selected_labels = labels.set_index("id").loc[sorted(selected_ids)]
assert selected_labels["english_label"].notna().all()
assert selected_labels["english_label"].str.strip().ne("").all()
source_labels = rankings.set_index("id").loc[selected_labels.index]
assert source_labels["short_name"].str.strip().equals(selected_labels["spanish_label"].str.strip())
assert source_labels["name"].str.strip().equals(selected_labels["spanish_proposal_wording"].str.strip())
english = labels.set_index("id")["english_label"]
top_proposals = pd.DataFrame({"Rank": np.arange(1, len(left) + 1),
    "Left-associated proposal": left["id"].map(english).to_numpy(),
    "Right-associated proposal": right["id"].map(english).to_numpy()})
display(top_proposals)
top_proposals.to_csv(OUT / "top_proposals.csv", index=False)
top_proposals.to_latex(OUT / "top_proposals.tex", index=False, escape=True, column_format="r p{5cm} p{5cm}")
top_proposal_data = pd.DataFrame({"rank": np.arange(1, len(left) + 1),
    "left_proposal_id": left["id"].to_numpy(), "left_rank_ratio": left["ranking_izquierda_sort"].to_numpy(),
    "right_proposal_id": right["id"].to_numpy(), "right_rank_ratio": right["ranking_derecha_sort"].to_numpy()})
display(top_proposal_data)
top_proposal_data.to_csv(OUT / "top_proposals_source_data.csv", index=False)

,Rank,Left-associated proposal,Right-associated proposal
0,1,New Constitution,Tax relief for vulnerable older adults
1,2,Constituent Assembly,Catastrophic Illnesses
2,3,Water,Drug Trafficking
3,4,Tax Progressivity,Parliamentary Salary
4,5,Human Rights,Electoral System
5,6,State-owned Lithium,Parliament Reduction
6,7,40 hours,Medication VAT
7,8,National Plebiscites,Fonasa Pharmacy
8,9,Indigenous Peoples,Older Adult Care
9,10,Minimum Wage,Technical and Vocational Education


,rank,left_proposal_id,left_rank_ratio,right_proposal_id,right_rank_ratio
0,1,23,0.013333,45,0.055556
1,2,37,0.056818,64,0.086957
2,3,6,0.065217,8,0.113208
3,4,85,0.133333,21,0.185185
4,5,57,0.170732,78,0.192308
5,6,41,0.234375,34,0.232877
6,7,74,0.314286,18,0.333333
7,8,88,0.324324,65,0.430769
8,9,40,0.352941,39,0.461538
9,10,87,0.368421,60,0.468085


## Method benchmarks

Read the supplied runtime, classification and coordinate-coverage results as text to retain their reported precision and the `NC` marker. The presentation label **DYNAMAP (flexible head)** identifies the evaluated specification. This label mapping does not alter the source CSVs, numerical cells, row order or benchmark experiments.

In [5]:
def benchmark_display_labels(frame):
    """Rename method headers only, without modifying the supplied numerical cells."""
    displayed = frame.copy()
    displayed.columns = [column.replace("DYNAMAP original", "DYNAMAP (flexible head)")
                         for column in frame.columns]
    if not displayed.columns.is_unique:
        raise ValueError("The display-label mapping produced duplicate columns.")
    np.testing.assert_array_equal(displayed.to_numpy(), frame.to_numpy())
    return displayed

for name in ["benchmark_runtime", "benchmark_accuracy", "benchmark_coverage"]:
    source = PROJECT / "data/benchmarks" / f"{name}.csv"
    frame = pd.read_csv(source, dtype=str, keep_default_na=False)
    assert len(frame) == (12 if name == "benchmark_accuracy" else 14)
    assert frame.Participants.is_unique
    assert all(value == "NC" or np.isfinite(float(value.replace(",", "")))
               for value in frame.to_numpy().ravel())
    frame = benchmark_display_labels(frame)
    display(frame)
    frame.to_csv(OUT / f"{name}.csv", index=False)
    if name == "benchmark_accuracy":
        publication = frame.set_index("Participants")
        publication.columns = pd.MultiIndex.from_tuples(
            [tuple(column.split(" | ")) for column in publication.columns])
        display(publication)
        publication.to_latex(OUT / f"{name}.tex", index=True, escape=True,
                             column_format="r" + "c" * 9, multicolumn_format="c")
    else:
        frame.to_latex(OUT / f"{name}.tex", index=False, escape=True, column_format="rccc")

,Participants,DYNAMAP (flexible head) (s),W-NOMINATE (s),MCMC (s)
0,10,1.51,0.07,0.64
1,50,3.36,0.36,2.63
2,100,4.20,0.85,5.05
3,250,12.12,2.01,12.53
4,500,14.60,3.77,25.23
5,"1,000",28.66,9.29,51.82
6,"2,000",102.16,26.61,109.33
7,"5,000",158.64,173.42,384.23
8,"10,000",305.59,"1,092.02","1,427.77"
9,"20,000",686.91,"7,815.78","7,819.64"


,Participants,MCMC | Logistic,MCMC | RF,MCMC | SVM,DYNAMAP (flexible head) | Logistic,DYNAMAP (flexible head) | RF,DYNAMAP (flexible head) | SVM,W-NOMINATE | Logistic,W-NOMINATE | RF,W-NOMINATE | SVM
0,10,33.3,33.3,33.3,33.3,66.7,33.3,33.3,66.7,33.3
1,50,46.7,73.3,40.0,26.7,53.3,26.7,73.3,86.7,73.3
2,100,63.3,66.7,60.0,63.3,60.0,63.3,66.7,70.0,63.3
3,250,77.3,73.3,78.7,64.0,60.0,54.7,76.0,68.9,74.7
4,500,84.0,84.0,82.7,82.6,80.5,86.6,82.2,83.7,83.1
5,"1,000",82.0,77.3,82.0,78.5,75.8,80.1,82.8,81.6,83.2
6,"2,000",86.0,86.8,86.2,85.2,82.3,85.3,84.7,83.2,85.4
7,"5,000",84.0,82.7,84.0,83.9,83.7,84.2,84.4,84.7,84.9
8,"10,000",56.1,53.1,56.1,83.6,83.3,83.9,56.9,64.4,57.0
9,"20,000",59.6,53.1,59.6,82.9,82.1,83.3,61.0,60.6,60.9


MCMC             DYNAMAP (flexible head)              \
             Logistic    RF   SVM                Logistic    RF   SVM   
Participants                                                            
10               33.3  33.3  33.3                    33.3  66.7  33.3   
50               46.7  73.3  40.0                    26.7  53.3  26.7   
100              63.3  66.7  60.0                    63.3  60.0  63.3   
250              77.3  73.3  78.7                    64.0  60.0  54.7   
500              84.0  84.0  82.7                    82.6  80.5  86.6   
1,000            82.0  77.3  82.0                    78.5  75.8  80.1   
2,000            86.0  86.8  86.2                    85.2  82.3  85.3   
5,000            84.0  82.7  84.0                    83.9  83.7  84.2   
10,000           56.1  53.1  56.1                    83.6  83.3  83.9   
20,000           59.6  53.1  59.6                    82.9  82.1  83.3   
22,227           62.0  57.9  62.0                    82.8  82.0  82.7   
30,000             NC    NC    NC                    83.3  82.4  83.3   

             W-NOMINATE              
               Logistic    RF   SVM  
Participants                         
10                 33.3  66.7  33.3  
50                 73.3  86.7  73.3  
100                66.7  70.0  63.3  
250                76.0  68.9  74.7  
500                82.2  83.7  83.1  
1,000              82.8  81.6  83.2  
2,000              84.7  83.2  85.4  
5,000              84.4  84.7  84.9  
10,000             56.9  64.4  57.0  
20,000             61.0  60.6  60.9  
22,227             63.5  57.5  63.5  
30,000               NC    NC    NC

,Participants,W-NOMINATE (%),MCMC (%),DYNAMAP (flexible head) (%)
0,10,20.0,0.0,0.0
1,50,12.0,0.0,0.0
2,100,10.0,0.0,0.0
3,250,7.6,0.0,0.0
4,500,7.0,0.0,0.0
5,"1,000",6.6,0.0,0.0
6,"2,000",4.9,0.0,0.0
7,"5,000",4.2,0.0,0.0
8,"10,000",3.7,0.0,0.0
9,"20,000",3.9,0.0,0.0


## External validation

All four specifications use the same eligible participants and ten folds within each country. Verify the sample identifiers, targets and folds against the individual prediction files. Accuracy and AUC are pooled out-of-fold quantities. Fold ranges are not confidence intervals.

In [6]:
validation = pd.read_csv(INPUTS["validation"])
country_order = ["Chile", "France", "Brazil"]
model_order = ["coordinates", "pc1", "demographics", "combined"]
predictor_labels = {"coordinates": "Two coordinates", "pc1": "PC1", "demographics": "Demographics",
                    "combined": "Coordinates + demographics"}
required = ["country", "model", "n", "oof_accuracy", "fold_accuracy_min", "fold_accuracy_max", "oof_auc"]
assert set(required).issubset(validation.columns)
assert len(validation) == len(country_order) * len(model_order)
assert set(zip(validation["country"], validation["model"])) == {(c, m) for c in country_order for m in model_order}
assert not validation.duplicated(["country", "model"]).any()
assert np.isfinite(validation[required[2:]]).all().all()
for metric in ["oof_accuracy", "fold_accuracy_min", "fold_accuracy_max", "oof_auc"]:
    assert validation[metric].between(0, 1).all()
assert (validation["fold_accuracy_min"] <= validation["fold_accuracy_max"]).all()
validation_rows = []
for country in country_order:
    prediction_directory = PROJECT / "outputs/validation" / country
    reference = pd.read_csv(prediction_directory / "coordinates_oof_predictions.csv", dtype={"participant_id": str})
    assert reference.participant_id.is_unique
    eligible_n = len(reference)
    country_rows = validation[validation["country"].eq(country)]
    assert country_rows["n"].eq(eligible_n).all() and eligible_n > 0
    for model_name in model_order:
        row = country_rows[country_rows["model"].eq(model_name)].iloc[0]
        predictions = pd.read_csv(prediction_directory / f"{model_name}_oof_predictions.csv", dtype={"participant_id": str})
        pd.testing.assert_frame_equal(predictions[["participant_id", "target", "fold"]],
                                      reference[["participant_id", "target", "fold"]])
        assert np.isclose(predictions.target.eq(predictions.prediction).mean(), row["oof_accuracy"], rtol=0, atol=1e-12)
        assert row["fold_accuracy_min"] - 1e-12 <= row["oof_accuracy"] <= row["fold_accuracy_max"] + 1e-12
        validation_rows.append({"Country": country, "N": eligible_n, "Predictors": predictor_labels[model_name],
            "Accuracy (%)": 100 * row["oof_accuracy"],
            "Fold range (%)": f"{100 * row['fold_accuracy_min']:.1f}--{100 * row['fold_accuracy_max']:.1f}",
            "AUC": row["oof_auc"]})
participant_validation = pd.DataFrame(validation_rows)
display(participant_validation)
participant_validation.to_csv(OUT / "participant_validation.csv", index=False)
participant_validation.to_latex(OUT / "participant_validation.tex", index=False, escape=True,
    column_format="lrlrrr", formatters={"Accuracy (%)": "{:.1f}".format, "AUC": "{:.3f}".format})

,Country,N,Predictors,Accuracy (%),Fold range (%),AUC
0,Chile,22024,Two coordinates,80.330548,79.0--81.6,0.867072
1,Chile,22024,PC1,74.387032,72.6--75.6,0.837743
2,Chile,22024,Demographics,56.116055,54.5--57.9,0.567266
3,Chile,22024,Coordinates + demographics,81.551943,80.1--83.2,0.879207
4,France,788,Two coordinates,91.243655,86.1--94.9,0.941808
5,France,788,PC1,87.944162,79.7--94.9,0.901953
6,France,788,Demographics,81.725888,81.0--82.3,0.542912
7,France,788,Coordinates + demographics,91.116751,86.1--94.9,0.945798
8,Brazil,495,Two coordinates,87.272727,81.6--93.9,0.931767
9,Brazil,495,PC1,84.242424,75.5--88.0,0.918571


## Geometry across initializations

Summarize the ten pairs of five fits. Distance correlations retain their signs. PC1 uses the absolute Spearman correlation because its direction is arbitrary. The endpoints are descriptive ranges.

In [7]:
geometry = pd.read_csv(INPUTS["geometry"])
geometry_specifications = [
    ("Proposal distances: Pearson", "Proposal distances", "Pearson"),
    ("Participant distances: Pearson", "Participant distances", "Pearson"),
    ("First component: absolute Spearman", "Participant PC1", "Absolute Spearman")]
assert {"measure", "fit_pairs", "minimum", "maximum"}.issubset(geometry.columns)
assert len(geometry) == len(geometry_specifications) and geometry["measure"].is_unique
assert set(geometry["measure"]) == {item[0] for item in geometry_specifications}
assert geometry["fit_pairs"].eq(5 * (5 - 1) // 2).all()
assert np.isfinite(geometry[["minimum", "maximum"]]).all().all()
assert geometry[["minimum", "maximum"]].ge(-1 - 1e-12).all().all()
assert geometry[["minimum", "maximum"]].le(1 + 1e-12).all().all()
assert (geometry["minimum"] <= geometry["maximum"]).all()
geometry_rows = []
for source_name, quantity, correlation in geometry_specifications:
    row = geometry.loc[geometry["measure"].eq(source_name)].iloc[0]
    geometry_rows.append({"Quantity": quantity, "Correlation": correlation,
                          "Minimum": row["minimum"], "Maximum": row["maximum"]})
geometry_across_fits = pd.DataFrame(geometry_rows)
display(geometry_across_fits)
geometry_across_fits.to_csv(OUT / "geometry_across_fits.csv", index=False)
geometry_across_fits.to_latex(OUT / "geometry_across_fits.tex", index=False, escape=True,
                             column_format="llrr", float_format="%.2f")

,Quantity,Correlation,Minimum,Maximum
0,Proposal distances,Pearson,0.501133,0.629081
1,Participant distances,Pearson,0.579881,0.689794
2,Participant PC1,Absolute Spearman,0.880589,0.912056


## All-cycle architecture comparison

Read the six supplied configurations without refitting them. Check that their sample and partition agree and that choice-parameter counts match the reported trainable tensors after excluding embeddings.

In [8]:
architectures = pd.read_csv(INPUTS["architectures"])
variant_order = ["simple_nom", "embedding_mlp", "structured_8", "structured_20", "nom_dense", "single_embedding"]
required = ["variant", "Architecture", "participants", "proposals", "records", "training_records",
            "test_records", "embedding_parameters", "Choice parameters", "Accuracy (%)", "split_sha256"]
assert set(required).issubset(architectures.columns)
assert len(architectures) == len(variant_order) and architectures["variant"].is_unique
assert set(architectures["variant"]) == set(variant_order)
assert architectures["Accuracy (%)"].between(0, 100).all()
assert architectures["Choice parameters"].ge(0).all()
for column in ["participants", "proposals", "records", "training_records", "test_records", "split_sha256"]:
    assert architectures[column].nunique() == 1, f"Architecture fits disagree on {column}."
assert (architectures["records"] == architectures["training_records"] + architectures["test_records"]).all()
assert (architectures["embedding_parameters"] == 2 * (architectures["participants"] + architectures["proposals"])).all()
for variant in variant_order:
    row = architectures.loc[architectures["variant"].eq(variant)].iloc[0]
    parameter_path = PROJECT / "data/benchmarks/architecture_parameters" / variant / "trainable_parameters.csv"
    assert parameter_path.is_file(), f"Missing trainable tensor report for {variant}."
    parameters = pd.read_csv(parameter_path)
    assert {"tensor", "shape", "parameters"}.issubset(parameters.columns)
    assert parameters["parameters"].ge(1).all()
    derived_choice_parameters = int(parameters["parameters"].sum()) - int(row["embedding_parameters"])
    assert derived_choice_parameters == row["Choice parameters"]
architecture_table = architectures.set_index("variant").loc[variant_order,
    ["Architecture", "Choice parameters", "Accuracy (%)"]].reset_index(drop=True)
architecture_table["Choice parameters"] = architecture_table["Choice parameters"].astype(int)
display(architecture_table)
architecture_table.to_csv(OUT / "historical_architectures.csv", index=False)
architecture_table.to_latex(OUT / "historical_architectures.tex", index=False, escape=True,
    column_format="p{8.8cm}rr", formatters={"Accuracy (%)": "{:.2f}".format})

,Architecture,Choice parameters,Accuracy (%)
0,Embedding + simple NOM-type head,6,65.472311
1,Embedding + MLP (512/64/4 units),36681,50.513059
2,"Structured head (8 units, 4 per alternative)",20,67.853534
3,"Structured head (20 units, 10 per alternative)",44,68.862689
4,NOM-DENSE + dense layers,61,68.881637
5,Single embedding table + structured head (8 un...,20,66.945541
